In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

df = yf.download("SPY", period="2y", interval="4h", multi_level_index=False)

class ATR:
    def __init__(self, df, period=14):
        self.df = df.copy()
        self.period = period

    def calculate(self):
        tr = np.maximum(self.df['High'] - self.df['Low'],
                        np.maximum(abs(self.df['High'] - self.df['Close'].shift(1)),
                                   abs(self.df['Low'] - self.df['Close'].shift(1))))
        return tr.ewm(com=self.period - 1, adjust=False).mean()

[*********************100%***********************]  1 of 1 completed


In [3]:
# cspell:words supertrend
class SuperTrend:
    def __init__(self, df, period=10, multiplier=3):
        self.df = df.copy()
        self.period = period
        self.multiplier = multiplier

    def calculate(self):
        direction = 1
        self.df['SuperTrend'] = np.nan
        st_loc = self.df.columns.get_loc('SuperTrend')
        self.df['ATR'] = ATR(self.df).calculate()
        for index, row in enumerate(self.df.itertuples()):
            if index < self.period:
                continue
            
            previous = self.df.iloc[index - 1, st_loc]
            if np.isnan(previous):
                previous = (row.High + row.Low) / 2
            
            if direction == 1:
                current = (row.High + row.Low) / 2 - row.ATR * self.multiplier
                self.df.iloc[index, st_loc] = current if current > previous else previous
                if row.Close < self.df.iloc[index, st_loc]:
                    direction = -1
            
            elif direction == -1:
                current = (row.High + row.Low) / 2 + row.ATR * self.multiplier
                self.df.iloc[index, st_loc] = current if current < previous else previous
                if row.Close > self.df.iloc[index, st_loc]:
                    direction = 1
        return self.df['SuperTrend']

supertrend = SuperTrend(df).calculate()
print(supertrend)

2024-06-12 09:30:00-04:00           NaN
2024-06-12 13:30:00-04:00           NaN
2024-06-13 09:30:00-04:00           NaN
2024-06-13 13:30:00-04:00           NaN
2024-06-14 09:30:00-04:00           NaN
                                ...    
2026-06-10 09:30:00-04:00    747.451252
2026-06-10 13:30:00-04:00    747.451252
2026-06-11 09:30:00-04:00    747.451252
2026-06-11 13:30:00-04:00    747.451252
2026-06-12 09:30:00-04:00    747.451252
Name: SuperTrend, Length: 995, dtype: float64


In [4]:
class ADX:
    def __init__(self, df, period=14):
        self.df = df.copy()
        self.period = period
    
    def calculate(self):
        self.df["+DM"] = (self.df["High"] - self.df["High"].shift(1)).clip(lower=0).rolling(self.period).mean()
        self.df["-DM"] = (self.df["Low"].shift(1) - self.df["Low"]).clip(lower=0).rolling(self.period).mean()
        self.df["+DI"] = 100 * self.df["+DM"] / ATR(self.df).calculate()
        self.df["-DI"] = 100 * self.df["-DM"] / ATR(self.df).calculate()
        self.df["DX"] = 100 * abs(self.df["+DI"] - self.df["-DI"]) / (self.df["+DI"] + self.df["-DI"])
        self.df["ADX"] = self.df["DX"].ewm(com=self.period-1, adjust=False).mean()
        return self.df["ADX"].dropna()

print(ADX(df).calculate())

2024-06-24 09:30:00-04:00    12.582781
2024-06-24 13:30:00-04:00    12.774090
2024-06-25 09:30:00-04:00    13.209378
2024-06-25 13:30:00-04:00    13.810683
2024-06-26 09:30:00-04:00    14.954094
                               ...    
2026-06-10 09:30:00-04:00    46.962252
2026-06-10 13:30:00-04:00    48.937557
2026-06-11 09:30:00-04:00    50.609081
2026-06-11 13:30:00-04:00    50.830801
2026-06-12 09:30:00-04:00    50.061315
Name: ADX, Length: 981, dtype: float64


In [5]:
class RSI:
    def __init__(self, df, period=14):
        self.df = df.copy()
        self.period = period
    
    def calculate(self):
        self.df["Delta"] = self.df['Close'].diff()
        self.df["Gain"] = self.df["Delta"].clip(lower=0)
        self.df["Loss"] = -self.df["Delta"].clip(upper=0)
        self.df["Avg_Gain"] = self.df["Gain"].ewm(com=self.period-1, adjust=False).mean()
        self.df["Avg_Loss"] = self.df["Loss"].ewm(com=self.period-1, adjust=False).mean()
        self.df["RS"] = self.df["Avg_Gain"] / self.df["Avg_Loss"]
        self.df["RSI"] = 100 - (100 / (1 + self.df["RS"]))
        return self.df["RSI"].dropna()
    
print(RSI(df).calculate())

2024-06-12 13:30:00-04:00     0.000000
2024-06-13 09:30:00-04:00     1.834582
2024-06-13 13:30:00-04:00     5.190467
2024-06-14 09:30:00-04:00     5.105998
2024-06-14 13:30:00-04:00     8.419677
                               ...    
2026-06-10 09:30:00-04:00    35.523368
2026-06-10 13:30:00-04:00    32.859602
2026-06-11 09:30:00-04:00    40.982292
2026-06-11 13:30:00-04:00    46.057494
2026-06-12 09:30:00-04:00    49.730880
Name: RSI, Length: 994, dtype: float64


In [6]:
# cspell:words dema
class DEMA:
    def __init__(self, df, period=20):
        self.df = df.copy()
        self.period = period
    
    def calculate(self):
        self.df['DEMA1'] = self.df['Close'].ewm(span=self.period, adjust=False).mean()
        self.df['DEMA2'] = self.df['DEMA1'].ewm(span=self.period, adjust=False).mean()
        return 2 * self.df['DEMA1'] - self.df['DEMA2']

In [7]:
print(SuperTrend(df).calculate().dropna())
print(ADX(df).calculate())
print(RSI(df).calculate())

2024-06-20 09:30:00-04:00    547.929993
2024-06-20 13:30:00-04:00    547.929993
2024-06-21 09:30:00-04:00    547.929993
2024-06-21 13:30:00-04:00    547.929993
2024-06-24 09:30:00-04:00    547.929993
                                ...    
2026-06-10 09:30:00-04:00    747.451252
2026-06-10 13:30:00-04:00    747.451252
2026-06-11 09:30:00-04:00    747.451252
2026-06-11 13:30:00-04:00    747.451252
2026-06-12 09:30:00-04:00    747.451252
Name: SuperTrend, Length: 985, dtype: float64
2024-06-24 09:30:00-04:00    12.582781
2024-06-24 13:30:00-04:00    12.774090
2024-06-25 09:30:00-04:00    13.209378
2024-06-25 13:30:00-04:00    13.810683
2024-06-26 09:30:00-04:00    14.954094
                               ...    
2026-06-10 09:30:00-04:00    46.962252
2026-06-10 13:30:00-04:00    48.937557
2026-06-11 09:30:00-04:00    50.609081
2026-06-11 13:30:00-04:00    50.830801
2026-06-12 09:30:00-04:00    50.061315
Name: ADX, Length: 981, dtype: float64
2024-06-12 13:30:00-04:00     0.000000
2024-06